In [1]:
from numba import jit


@jit
def g(x):
    return x * x


@jit
def f(x):
    return g(x) + 1

In [2]:
x = 3.0
print("f(x):", f(x))
print("g(x):", g(x))
print("f signatures:", f.signatures)
print("g signatures:", g.signatures)

f(x): 10.0
g(x): 9.0
f signatures: [(float64,)]
g signatures: [(float64,)]


In [25]:
for i, overload in enumerate(f.overloads.values()):
    print(f"Overload {i}:")
    print(f"  signature: {f.signatures[i]}")
    print(f"  target context: {overload.target_context}")
    for name in dir(overload.target_context):
        try:
            value = getattr(overload.target_context, name)
        except Exception as err:
            value = err
        print(f"    {name}: {value}")

Overload 0:
  signature: (float64,)
  target context: <numba.core.cpu.CPUContext object at 0x1174cead0>
    DIBuilder: <class 'numba.core.debuginfo.DIBuilder'>
    __class__: <class 'numba.core.cpu.CPUContext'>
    __delattr__: <method-wrapper '__delattr__' of CPUContext object at 0x1174cead0>
    __dict__: {'address_size': 64, 'typing_context': <numba.core.typing.context.Context object at 0x116148d70>, 'target_name': 'cpu', 'target': <class 'numba.core.target_extension.CPU'>, '_registries': {Lowering Registry<cmathimpl>: <numba.core.imputils.RegistryLoader object at 0x117500ad0>, Lowering Registry<cffiimpl>: <numba.core.imputils.RegistryLoader object at 0x1174ce5d0>, Lowering Registry<mathimpl>: <numba.core.imputils.RegistryLoader object at 0x1174ce710>, Lowering Registry<npyimpl>: <numba.core.imputils.RegistryLoader object at 0x1175983e0>, Lowering Registry<printimpl>: <numba.core.imputils.RegistryLoader object at 0x117598510>, Lowering Registry<randomimpl>: <numba.core.imputils.Regi

In [20]:
overload.target_context.codegen()

In [21]:
f_result = f.overloads[f.signatures[0]]
g_result = g.overloads[g.signatures[0]]
f_context = f_result.target_context
g_context = g_result.target_context
f_codegen = f_context.codegen()
g_codegen = g_context.codegen()

### 5. Compare them
`True` means both references point to one object. `False` means separate objects.

In [24]:
print("f is g\t\t", f is g)
print("typingctx\t", f.typingctx is g.typingctx)
print("cpu-context\t", f.targetctx is g.targetctx)
print("compilation result\t", f_result is g_result)
print("compilation cpu-context\t", f_context is g_context)
print("code-library\t", f_result.library is g_result.library)
print("llvm module\t", f_result.library._final_module is g_result.library._final_module)
print("codegen\t\t", f_codegen is g_codegen)
print("target machine\t", f_codegen._tm is g_codegen._tm)
print("jit engine\t", f_codegen._engine is g_codegen._engine)

f is g		 False
typingctx	 True
cpu-context	 True
compilation result	 False
compilation cpu-context	 False
code-library	 False
llvm module	 False
codegen		 True
target machine	 True
jit engine	 True


In [26]:
f_result.library._linking_libraries

[<Library 'g' at 0x117710f50>, <Library 'nrt' at 0x116212660>]